# Step 12 - Computational & Deployment Analysis

Measure training time, prediction latency/throughput, peak fit memory, and serialized model size for every model, then assess feasibility for resource-constrained IoT edge devices.

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

def _find_root():
    p = Path.cwd().resolve()
    for cand in [p, *p.parents]:
        if (cand / "src").is_dir() and (cand / "data").is_dir():
            return cand
    return p

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src import config as C
from src import viz
viz.setup_style()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("Repository root:", ROOT)


Repository root: /Users/rauankaztaev/IdeaProjects/Draft/project


In [2]:
from src import data, models, computational as comp
from src import utils

df = data.load_clean()
x_train, x_test, y_train, y_test = data.get_splits(df)
zoo = models.get_model_zoo(x_train)
NAMES = list(zoo.keys())
profile = comp.profile_models(zoo, x_train, y_train, x_test, y_test, NAMES)
display(profile)
utils.save_table(profile, "computational_profile",
                 caption="Training/inference cost and model size.", label="tab:compute")

,Model,Fit Time (s),Predict Time (s),Latency (ms/sample),Throughput (samples/s),Peak Fit Memory (MB),Model Size (KB)
0,Logistic Regression,0.0307,0.00241,0.00148,674362.7,1.371,2.89
1,Ridge Classifier,0.0231,0.00221,0.00136,735356.9,1.614,2.89
2,GaussianNB,0.0134,0.00155,0.00095,1049370.5,1.270,2.95
3,SGD Classifier,0.0463,0.00288,0.00177,563546.6,1.275,3.12
4,Decision Tree,0.0199,0.00327,0.00201,496833.0,1.274,8.92
5,AdaBoost,0.1747,0.00537,0.00331,302277.3,1.269,30.64
6,SVM,0.4513,0.03453,0.02128,46997.2,1.275,59.96
7,MLP,3.3348,0.00373,0.00230,435001.1,1.272,71.45
8,Gradient Boosting,0.3206,0.00341,0.00210,476093.8,1.295,134.75
9,CatBoost,0.4970,0.00237,0.00146,684054.8,1.273,440.11


{'csv': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/computational_profile.csv'),
 'tex': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/computational_profile.tex')}

## 10.1 Deployment feasibility view

In [3]:
deploy = comp.deployment_view(profile, size_budget_kb=500.0, latency_budget_ms=1.0)
display(deploy[["Model", "Model Size (KB)", "Latency (ms/sample)", "edge_feasible"]])
utils.save_table(deploy, "deployment_feasibility",
                 caption="Edge-deployment feasibility (500 KB / 1 ms budget).", label="tab:deploy")

,Model,Model Size (KB),Latency (ms/sample),edge_feasible
0,Logistic Regression,2.89,0.00148,True
1,Ridge Classifier,2.89,0.00136,True
2,GaussianNB,2.95,0.00095,True
3,SGD Classifier,3.12,0.00177,True
4,Decision Tree,8.92,0.00201,True
5,AdaBoost,30.64,0.00331,True
6,SVM,59.96,0.02128,True
7,MLP,71.45,0.00230,True
8,Gradient Boosting,134.75,0.00210,True
9,CatBoost,440.11,0.00146,True


{'csv': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/deployment_feasibility.csv'),
 'tex': PosixPath('/Users/rauankaztaev/IdeaProjects/Draft/project/tables/deployment_feasibility.tex')}

## 10.2 Cost/size visualisations

In [4]:
import seaborn as sns
fig, ax = plt.subplots(figsize=(7.5, 5))
sns.scatterplot(data=profile, x="Model Size (KB)", y="Latency (ms/sample)",
                hue="Model", s=90, ax=ax, legend=False)
for _, r in profile.iterrows():
    ax.annotate(r["Model"], (r["Model Size (KB)"], r["Latency (ms/sample)"]),
                fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_title("Model size vs inference latency (log-log)")
fig.tight_layout(); fig.savefig(C.FIGURES_DIR / "compute_size_latency.png", dpi=300, bbox_inches="tight"); plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(x="Fit Time (s)", y="Model", data=profile.sort_values("Fit Time (s)"),
            ax=ax, hue="Model", legend=False)
ax.set_title("Training time by model")
fig.tight_layout(); fig.savefig(C.FIGURES_DIR / "compute_train_time.png", dpi=300, bbox_inches="tight"); plt.close(fig)
print("Saved compute figures.")

Saved compute figures.


**Interpretation.** Random Forest / Extra Trees produce the largest serialized models (hundreds of trees), while a single Decision Tree, Logistic Regression, and GaussianNB are tiny and fastest. Gradient boosters sit in between with excellent latency. For an IoT gateway, a boosted model or a depth-limited forest offers the best accuracy/size trade-off; the linear model is the fallback for the most constrained microcontrollers. Sub-millisecond per-sample latency makes all models viable for the low sampling rates typical of cold-storage telemetry.